# Pipeline OCR → HTML com OpenRouter
OCR com PaddleOCR (`pipeline.ocr`), HTML gerado por LLM na OpenRouter (`pipeline.render`) e métricas por etapa (`pipeline.metrics`). A chave vem de `OPENROUTER_API_KEY` ou do `.env`.

**No Colab:** guarde `OPENROUTER_API_KEY` em Secrets (🔑) e envie o `colab.zip` quando pedido.

In [1]:
MODEL = "qwen/qwen3.8-27b:free"
IMAGE = "input/amostra_diversa/captura/300/01-1-cinza-300/p01.png"
KINDS = ("compact",)  # entradas do llm: compact, markdown, full_json
OUTPUT = "output/notebook"
CPU_THREADS = 2
REASONING = False  # ligado, o qwen3.8 gasta 5–16 mil tokens pensando e pode truncar

In [2]:
import os, sys
from pathlib import Path

if "google.colab" in sys.modules:
    ROOT = Path("/content/projeto")
    if not (ROOT / "pipeline").is_dir():
        import zipfile
        from google.colab import files
        zipfile.ZipFile(next(iter(files.upload()))).extractall(ROOT)
    from google.colab import userdata
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
    !pip install -q paddleocr==3.7.0 paddlepaddle==3.3.1 "paddlex[ocr]==3.7.2" openai python-dotenv beautifulsoup4 opencv-contrib-python psutil
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.environ.setdefault("PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK", "True")

import pandas as pd
from IPython.display import IFrame, display
from pipeline import metrics
from pipeline.ocr import load_engine, process_file
from pipeline.render import output_path, write_html

image, output = ROOT / IMAGE, ROOT / OUTPUT
show = lambda prefix: display(pd.DataFrame([r for r in metrics.summary() if r["stage"].startswith(prefix)]).round(2))

## 1. OCR

In [3]:
with metrics.stage("ocr:carregar_modelo"):
    engine = load_engine(cpu_threads=CPU_THREADS)
folder, status = process_file(engine, image, image.parent, output)
layout = next(folder.glob("*_layout.json"))
print(status, layout)
show("ocr")

/home/larissa/Downloads/InvoiceConverterPoC-NLP-26.2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


which: no ccache in (/home/larissa/.config/Code - OSS/User/globalStorage/github.copilot-chat/debugCommand:/home/larissa/.config/Code - OSS/User/globalStorage/github.copilot-chat/copilotCli:/home/larissa/.local/bin:/home/larissa/.config/Code - OSS/User/globalStorage/github.copilot-chat/debugCommand:/home/larissa/.config/Code - OSS/User/globalStorage/github.copilot-chat/copilotCli:/usr/local/bin:/usr/bin:/usr/local/sbin:/usr/bin/site_perl:/usr/bin/vendor_perl:/usr/bin/core_perl)
/home/larissa/Downloads/InvoiceConverterPoC-NLP-26.2/.venv/lib/python3.12/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)


Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/larissa/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.


Creating model: ('PP-DocBlockLayout', None, None)


...

Creating model: ('latin_PP-OCRv5_mobile_rec', None, None)


Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/larissa/.paddlex/official_models/latin_PP-OCRv5_mobile_rec`.


/home/larissa/Downloads/InvoiceConverterPoC-NLP-26.2/pipeline/ocr.py:131: RuntimeWarning: overflow encountered in scalar multiply
  return max(0, min(x2, c) - max(x1, a)) * max(0, min(y2, d) - max(y1, b)) / max(1, (x2-x1)*(y2-y1))


ok /home/larissa/Downloads/InvoiceConverterPoC-NLP-26.2/output/intermediarios/notebook/p01_png/p01_page_001_layout.json


,stage,calls,seconds,cpu_%,peak_ram_mb,gpu_%,peak_vram_mb,tokens_in,tokens_out,cost_usd
0,ocr:carregar_modelo,1,7.94,113.20,1549.01,None,None,0,0,0.0
1,ocr:preparar_imagem,1,0.01,144.73,1549.03,None,None,0,0,0.0
2,ocr:predict,1,58.22,190.67,2042.95,None,None,0,0,0.0
3,ocr:reparar_tabelas,1,0.16,172.25,2044.83,None,None,0,0,0.0
4,ocr:salvar,1,0.25,95.31,2045.80,None,None,0,0,0.0


## 2. HTML via OpenRouter

In [4]:
logs = [write_html(layout, kind, output, engine="openrouter", model=MODEL,
                   reasoning=REASONING) for kind in KINDS]
display(pd.DataFrame(logs))
show("html")

HTML (compact): /home/larissa/Downloads/InvoiceConverterPoC-NLP-26.2/output/html/notebook/p01_png/p01_page_001_html.html


,source,engine,status,model,tokens_in,tokens_out,cost_usd,duration_seconds
0,/home/larissa/Downloads/InvoiceConverterPoC-NL...,openrouter,accepted,qwen/qwen3.8-27b:free,2165,3936,0.0,21.531


,stage,calls,seconds,cpu_%,peak_ram_mb,gpu_%,peak_vram_mb,tokens_in,tokens_out,cost_usd
0,html:openrouter:compact,1,21.53,5.99,2041.14,None,None,2165,3936,0.0


In [5]:
IFrame(os.path.relpath(output_path(layout, KINDS[0], output)), width="100%", height=600)

## 3. Resumo

In [6]:
metrics.report("notebook");

                  stage  calls  seconds  cpu_%  peak_ram_mb  gpu_%  peak_vram_mb  tokens_in  tokens_out  cost_usd
    ocr:carregar_modelo      1      7.9  113.2       1549.0      -             -          0           0    0.0000
    ocr:preparar_imagem      1   0.0069  144.7       1549.0      -             -          0           0    0.0000
            ocr:predict      1     58.2  190.7       2042.9      -             -          0           0    0.0000
    ocr:reparar_tabelas      1   0.1567  172.2       2044.8      -             -          0           0    0.0000
             ocr:salvar      1   0.2518   95.3       2045.8      -             -          0           0    0.0000
html:openrouter:compact      1     21.5    6.0       2041.1      -             -       2165        3936    0.0000
Métricas: /home/larissa/Downloads/InvoiceConverterPoC-NLP-26.2/runs/metrics_20261003_170101_713441.json


In [7]:
if "google.colab" in sys.modules:  # baixa output/ e runs/ num zip
    !cd {ROOT} && zip -qr /content/colab_resultados.zip output runs
    from google.colab import files
    files.download("/content/colab_resultados.zip")